# 대조 실험 C — 가상 염색 없이, 비염색 그대로 YOLO 추가 학습

**위에서부터 `Shift + Enter`만. 고칠 값 없음.** 약 30분~1시간.

B(가상 염색 영상으로 YOLO 추가 학습)가 RBC F1 0.954로 크게 좋아졌는데, 이게 **가상 염색 덕분인지**, 아니면 **RedTell 정답으로 추가 학습한 덕분인지** 아직 구분이 안 됩니다.
C는 B와 **모든 조건이 같고 입력만 다릅니다** (가상 염색 영상 대신 비염색 원본).

| | 추가 학습 데이터 | 채점 입력 |
| --- | --- | --- |
| B | TXL 염색 + **가상 염색한** RedTell dse 1~83 | 가상 염색한 RedTell test |
| **C** | TXL 염색 + **비염색 원본** RedTell dse 1~83 | 비염색 원본 RedTell test |

읽는 법: C ≈ B → 개수 정확도의 개선은 주로 RedTell 정답으로 추가 학습한 효과 (가상 염색은 사람이 보기 쉬운 영상을 주는 역할). C가 B보다 확실히 낮음 → 가상 염색이 탐지에 도움이 됨.

In [1]:
import torch, os, sys, json, pandas as pd
os.chdir(r'D:\05. Blood\05. Train\code')
P = r"D:/05. Blood/04. Prep"; T = r"D:/05. Blood/05. Train"; R6 = T + "/results/6_detector"
YOLO_RAW = T + "/runs/yolo/yolo11s_ft_raw/weights/best.pt"
if not os.path.exists(YOLO_RAW):
    !{sys.executable} 24_finetune_yolo.py --prep "{P}" --train_root "{T}" --vs_train "{P}/6_redtell_dse_train/images" --vs_val "{P}/4_eval_unstained/redtell_dse_val/images" --name yolo11s_ft_raw --epochs 30

�뜲�씠�꽣: 媛��긽 �뿼�깋 �븰�뒿 82�옣 횞 4 + TXL �븰�뒿 �쟾泥�, 寃�利� 9�옣 �넂 D:/05. Blood/05. Train\datasets\yolo_ft_yolo11s_ft_raw\data.yaml
New https://pypi.org/project/ultralytics/8.4.203 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.184  Python-3.11.16 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16283MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:/05. Blood/05. Train\datasets\yolo_ft_yolo11s_ft_raw\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, h

## conf를 val로 고르고 test 한 번 (B와 같은 규칙)

In [2]:
CONFS = [0.5, 0.6, 0.7, 0.8, 0.9]
for c in CONFS:
    out = f"{R6}/raw_conf{c}_val"
    if not os.path.exists(out + "/summary.csv"):
        !{sys.executable} 12_eval_compare.py --prep "{P}" --images "{P}/4_eval_unstained/redtell_dse_val/images" --labels "{P}/4_eval_unstained/redtell_dse_val/labels" --yolo "{YOLO_RAW}" --conf {c} --out "{out}"
vt = pd.DataFrame([dict(conf=c, **pd.read_csv(f"{R6}/raw_conf{c}_val/summary.csv").iloc[0][["RBC_MAE", "RBC_precision", "RBC_recall", "RBC_F1"]]) for c in CONFS]); display(vt)
C_BEST = float(vt.sort_values(["RBC_F1", "RBC_MAE"], ascending=[False, True]).iloc[0].conf); print("val로 고른 conf:", C_BEST)
for kind in ["test", "txl_test"]:
    out = f"{R6}/raw_conf{C_BEST}_{kind}"
    if os.path.exists(out + "/summary.csv"):
        continue
    if kind == "test":
        !{sys.executable} 12_eval_compare.py --prep "{P}" --images "{P}/4_eval_unstained/redtell_dse/images" --labels "{P}/4_eval_unstained/redtell_dse/labels" --yolo "{YOLO_RAW}" --conf {C_BEST} --out "{out}"
    else:
        !{sys.executable} 12_eval_compare.py --prep "{P}" --yolo "{YOLO_RAW}" --conf {C_BEST} --out "{out}"


[YOLO]
  method: YOLO
  n_images: 9
  WBC_MAE: 0.0
  WBC_precision: 0.0
  WBC_recall: 0.0
  WBC_F1: 0.0
  RBC_MAE: 3.556
  RBC_precision: 0.9108
  RBC_recall: 0.997
  RBC_F1: 0.952
  WBC_rel_error: 
  RBC_rel_error: 0.0912
  ms_per_image: 7.9

����옣: D:/05. Blood/05. Train/results/6_detector/raw_conf0.5_val

[YOLO]
  method: YOLO
  n_images: 9
  WBC_MAE: 0.0
  WBC_precision: 0.0
  WBC_recall: 0.0
  WBC_F1: 0.0
  RBC_MAE: 2.778
  RBC_precision: 0.9174
  RBC_recall: 0.9852
  RBC_F1: 0.9501
  WBC_rel_error: 
  RBC_rel_error: 0.0684
  ms_per_image: 8.1

����옣: D:/05. Blood/05. Train/results/6_detector/raw_conf0.6_val

[YOLO]
  method: YOLO
  n_images: 9
  WBC_MAE: 0.0
  WBC_precision: 0.0
  WBC_recall: 0.0
  WBC_F1: 0.0
  RBC_MAE: 1.889
  RBC_precision: 0.9427
  RBC_recall: 0.9734
  RBC_F1: 0.9578
  WBC_rel_error: 
  RBC_rel_error: 0.0506
  ms_per_image: 8.0

����옣: D:/05. Blood/05. Train/results/6_detector/raw_conf0.7_val

[YOLO]
  method: YOLO
  n_images: 9
  WBC_MAE: 0.0
  WBC_precisio

,conf,RBC_MAE,RBC_precision,RBC_recall,RBC_F1
0,0.5,3.556,0.9108,0.9970,0.9520
1,0.6,2.778,0.9174,0.9852,0.9501
2,0.7,1.889,0.9427,0.9734,0.9578
3,0.8,1.111,0.9699,0.9527,0.9612
4,0.9,7.778,0.9813,0.7781,0.8680


val로 고른 conf: 0.8

[YOLO]
  method: YOLO
  n_images: 91
  WBC_MAE: 0.0
  WBC_precision: 0.0
  WBC_recall: 0.0
  WBC_F1: 0.0
  RBC_MAE: 1.901
  RBC_precision: 0.9617
  RBC_recall: 0.9576
  RBC_F1: 0.9596
  WBC_rel_error: 
  RBC_rel_error: 0.0498
  ms_per_image: 9.8

����옣: D:/05. Blood/05. Train/results/6_detector/raw_conf0.8_test

[YOLO]
  method: YOLO
  n_images: 126
  WBC_MAE: 0.048
  WBC_precision: 1.0
  WBC_recall: 0.9542
  WBC_F1: 0.9766
  RBC_MAE: 1.365
  RBC_precision: 0.9415
  RBC_recall: 0.9058
  RBC_F1: 0.9233
  WBC_rel_error: 0.0476
  RBC_rel_error: 0.1058
  ms_per_image: 9.2

����옣: D:/05. Blood/05. Train/results/6_detector/raw_conf0.8_txl_test


## B vs C 비교표

In [3]:
cols = ["RBC_MAE", "RBC_precision", "RBC_recall", "RBC_F1", "WBC_F1"]
B_BEST = json.load(open(R6 + "/selected_conf.json"))["ft"]
tab = {"B: 가상 염색 + 추가 학습 YOLO (RedTell test)": f"{R6}/ft_conf{B_BEST}_test",
       "C: 비염색 그대로 + 추가 학습 YOLO (RedTell test)": f"{R6}/raw_conf{C_BEST}_test",
       "B 모델, 진짜 염색 TXL test": f"{R6}/ft_conf{B_BEST}_txl_test",
       "C 모델, 진짜 염색 TXL test": f"{R6}/raw_conf{C_BEST}_txl_test"}
t = pd.DataFrame({k: pd.read_csv(v + "/summary.csv").iloc[0][cols] for k, v in tab.items()}).T
display(t); t.to_csv(R6 + "/compare_B_vs_C.csv", encoding="utf-8-sig")

,RBC_MAE,RBC_precision,RBC_recall,RBC_F1,WBC_F1
B: 가상 염색 + 추가 학습 YOLO (RedTell test),2.484,0.9385,0.969,0.9535,0.0
C: 비염색 그대로 + 추가 학습 YOLO (RedTell test),1.901,0.9617,0.9576,0.9596,0.0
"B 모델, 진짜 염색 TXL test",1.317,0.916,0.9496,0.9325,0.9962
"C 모델, 진짜 염색 TXL test",1.365,0.9415,0.9058,0.9233,0.9766
